In [37]:
import csv
import shutil
from pathlib import Path
import json
from jsonschema import ValidationError, SchemaError, validate, Draft7Validator
from collections import Counter




## Save pathnames of all v12 ai-project-json files

In [18]:

def filter_json_to_csv(
    root_directory: str,
    target_field: str,
    target_value: str | int | float | bool,
    output_csv_path: str,
) -> int:
    """Recursively searches for 'ai_*.json' files where a specific field equals a target value

    and writes their file paths to a CSV.

    :param root_directory: The root folder path to start searching from.
    :param target_field: The key/field name inside the JSON object to check.
    :param target_value: The expected value for the target field.
    :param output_csv_path: Path where the output CSV file should be saved.
    :return: The total count of matching files found.
    """
    root_path = Path(root_directory)
    matching_files = []

    # `rglob` recursively searches all nested folders for matching filenames
    for file_path in root_path.rglob("ai_*.json"):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

                # Ensure the JSON content is a dictionary before checking keys
                if (
                    isinstance(data, dict)
                    and data.get(target_field) == target_value
                ):
                    matching_files.append([str(file_path.resolve())])

        except (json.JSONDecodeError, OSError) as e:
            # Safely skip corrupted JSONs or unreadable files
            print(f"Skipping file due to error ({file_path}): {e}")

    # Write matches to CSV
    with open(
        output_csv_path, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["file_path"])  # Header column
        writer.writerows (matching_files)

    print(
        f"Done! Found {len(matching_files)} matching files out of all checked."
    )
    return len(matching_files)

In [19]:
proj_dump_dir = '/home/hemduttdabral/Downloads/proj-api-volume-dump'

filter_json_to_csv(
    root_directory=proj_dump_dir,
    target_field="schema_version",
    target_value="v12",
    output_csv_path="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_projs.csv")

Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_700039/ai_new_project_name_700039.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_164926/ai_new_project_name_164926.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_449184/ai_new_project_name_449184.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_467444/ai_new_project_name_467444.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/pr

111

In [20]:
## copy all v12 files to a new folder

In [21]:
def copy_and_rename_from_csv(
    csv_file_path: str, destination_dir: str, start_index: int = 1
) -> int:
    """Reads file paths from a CSV, copies them to a destination directory,

    and renames them using the pattern 'ai_v12_{i}.json'.

    :param csv_file_path: Path to the CSV file generated previously.
    :param destination_dir: Directory where copied files will be placed.
    :param start_index: Starting serial number for 'i' (defaults to 1).
    :return: Total number of files successfully copied.
    """
    dest_path = Path(destination_dir)
    dest_path.mkdir(
        parents=True, exist_ok=True
    )  # Ensures target directory exists

    copied_count = 0

    with open(csv_file_path, "r", encoding="utf-8") as csv_file:
        reader = csv.reader(csv_file)

        # Skip header if present ('file_path')
        header = next(reader, None)
        if header and header[0] != "file_path":
            # If the first row wasn't the header, reset file pointer
            csv_file.seek(0)

        for i, row in enumerate(reader, start=start_index):
            if not row:
                continue

            src_file_path = Path(row[0].strip())

            if not src_file_path.exists():
                print(f"Skipping (File not found): {src_file_path}")
                continue

            # Keep original extension (.json)
            extension = src_file_path.suffix or ".json"
            new_filename = f"ai_v12_{i}{extension}"
            target_path = dest_path / new_filename

            try:
                shutil.copy2(src_file_path, target_path)
                copied_count += 1
            except OSError as e:
                print(f"Failed to copy {src_file_path}: {e}")

    print(
        f"Successfully copied and renamed {copied_count} files into '{destination_dir}'."
    )
    return copied_count

In [22]:
copy_and_rename_from_csv(
    csv_file_path="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_projs.csv",
    destination_dir="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_projs_copied",
    start_index=0)

Successfully copied and renamed 111 files into '/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_projs_copied'.


111

## now check names of files which which do not conform to v12 schema

In [30]:
def find_invalid_jsons_to_csv(
    root_directory: str,
    schema: dict | str,
    output_csv_path: str,
    filename_prefix: str = "",
) -> int:
    """Recursively scans for .json files starting with a given prefix, validates each

    against a JSON schema, and writes non-conforming file paths to a CSV.

    :param root_directory: Directory path to scan recursively.
    :param schema: Schema dictionary or path/string to a schema .json file.
    :param output_csv_path: Output path for the generated CSV file.
    :param filename_prefix: Filter to only check files starting with this
    prefix (e.g., 'ai_').
    :return: Total count of non-conforming JSON files found.
    """
    # Load schema if given a file path string or Path object
    if isinstance(schema, (str, Path)):
        with open(schema, "r", encoding="utf-8") as f:
            schema = json.load(f)

    root_path = Path(root_directory)
    invalid_files = []

    # Dynamic file match pattern based on prefix
    pattern = f"{filename_prefix}*.json"

    for file_path in root_path.rglob(pattern):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Validate instance structure against schema
            validate(instance=data, schema=schema)

        except (ValidationError, json.JSONDecodeError) as e:
            error_reason = (
                e.message
                if isinstance(e, ValidationError)
                else "Invalid JSON syntax"
            )
            invalid_files.append([str(file_path.resolve()), error_reason])

        except OSError as e:
            print(f"Skipping unreadable file ({file_path}): {e}")

    # Save invalid file paths to CSV
    with open(
        output_csv_path, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["file_path", "error_reason"])
        writer.writerows(invalid_files)

    print(
        f"Scan complete. Found {len(invalid_files)} non-conforming '{pattern}' files."
    )
    return len(invalid_files)

In [31]:
find_invalid_jsons_to_csv(
    root_directory="/home/hemduttdabral/Downloads/proj-api-volume-dump",
    schema="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_master_schema.json",
    output_csv_path="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_invalid_jsons.csv",
    filename_prefix="ai_"
)

Scan complete. Found 1622 non-conforming 'ai_*.json' files.


1622

# Do a more lienent checking. i.e. if a required field is not present in the json file, consider that as a pass, and document the occurrence in a CSV file.

In [33]:
def find_invalid_jsons_lenient(
    root_directory: str,
    schema: dict | str,
    critical_errors_csv: str,
    missing_fields_csv: str,
    filename_prefix: str = "",
) -> tuple[int, int]:
    """Recursively validates JSON files against a schema, separating critical

    structural/type errors from missing required fields into two CSV files.

    :param root_directory: Directory path to scan recursively.
    :param schema: Schema dictionary or path/string to a schema .json file.
    :param critical_errors_csv: CSV path for hard failures (type mismatches, bad
        syntax).
    :param missing_fields_csv: CSV path for lenient failures (missing required
        keys).
    :param filename_prefix: Prefix filter for JSON filenames (e.g., 'ai_').
    :return: A tuple of (critical_error_count, missing_field_count).
    """
    # Load schema if passed as a file path
    if isinstance(schema, (str, Path)):
        with open(schema, "r", encoding="utf-8") as f:
            schema = json.load(f)

    root_path = Path(root_directory)
    critical_errors = []
    missing_fields = []

    pattern = f"{filename_prefix}*.json"

    for file_path in root_path.rglob(pattern):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            validate(instance=data, schema=schema)

        except ValidationError as e:
            abs_path = str(file_path.resolve())

            # Check if failure is ONLY due to a missing required field
            if e.validator == "required":
                missing_fields.append([abs_path, e.message])
            else:
                critical_errors.append([abs_path, e.message])

        except json.JSONDecodeError:
            critical_errors.append(
                [str(file_path.resolve()), "Invalid JSON syntax"]
            )

        except OSError as e:
            print(f"Skipping unreadable file ({file_path}): {e}")

    # Write critical structural errors to CSV
    with open(
        critical_errors_csv, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["file_path", "error_reason"])
        writer.writerows(critical_errors)

    # Write missing field cases to CSV
    with open(
        missing_fields_csv, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["file_path", "missing_field_info"])
        writer.writerows(missing_fields)

    print(f"Scan complete for pattern '{pattern}':")
    print(f" - Critical Errors logged: {len(critical_errors)}")
    print(f" - Missing Fields logged: {len(missing_fields)}")

    return len(critical_errors), len(missing_fields)

In [34]:
find_invalid_jsons_lenient(
    root_directory="/home/hemduttdabral/Downloads/proj-api-volume-dump",
    schema="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_master_schema.json",
    critical_errors_csv="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_critical_errors.csv",
    missing_fields_csv="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_missing_fields.csv",
    filename_prefix="ai_")

Scan complete for pattern 'ai_*.json':
 - Critical Errors logged: 64
 - Missing Fields logged: 1558


(64, 1558)

## Now keep track of missing keys for v12_missing_fields.csv file

Now modify above function, so that for all files in missing_field_csv, we record the fields which are missing in ai_*.json files, and we output the keys of these fields in a separate csv file. This csv should clearly mention count of files in which a given field is missing.

In [39]:
def analyze_missing_fields_to_csv(
    root_directory: str,
    schema: dict | str,
    critical_errors_csv: str,
    missing_fields_csv: str,
    field_summary_csv: str,
    filename_prefix: str = "ai_",
) -> tuple[int, int, int]:
    """Scans JSON files, validates against a schema, and generates:

    1. Critical errors CSV (type/syntax failures)
    2. Missing fields per file CSV
    3. Missing field counts summary CSV

    :return: Tuple of (critical_error_count, missing_field_file_count,
        unique_missing_fields_count)
    """
    # Load schema if path/string is provided
    if isinstance(schema, (str, Path)):
        with open(schema, "r", encoding="utf-8") as f:
            schema = json.load(f)

    validator = Draft7Validator(schema)
    root_path = Path(root_directory)

    critical_errors = []
    missing_fields_per_file = []
    missing_fields_counter = Counter()

    pattern = f"{filename_prefix}*.json"

    for file_path in root_path.rglob(pattern):
        abs_path = str(file_path.resolve())

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Collect ALL errors for this file
            errors = list(validator.iter_errors(data))

            if not errors:
                continue

            file_has_critical = False
            file_missing_keys = set()

            for error in errors:
                if error.validator == "required":
                    # Extracts the missing property name from the validation message
                    # e.g., "'status' is a required property" -> 'status'
                    missing_key = error.message.split("'")[1]
                    file_missing_keys.add(missing_key)
                else:
                    file_has_critical = True
                    critical_errors.append([abs_path, error.message])

            # If it had missing keys, log file-level details and update global counts
            if file_missing_keys:
                missing_fields_per_file.append(
                    [abs_path, ", ".join(sorted(file_missing_keys))]
                )
                missing_fields_counter.update(file_missing_keys)

        except json.JSONDecodeError:
            critical_errors.append([abs_path, "Invalid JSON syntax"])
        except OSError as e:
            print(f"Skipping unreadable file ({file_path}): {e}")

    # 1. Output Critical Errors CSV
    with open(
        critical_errors_csv, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["file_path", "error_reason"])
        writer.writerows(critical_errors)

    # 2. Output Per-File Missing Fields CSV
    with open(
        missing_fields_csv, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["file_path", "missing_fields"])
        writer.writerows(missing_fields_per_file)

    # 3. Output Aggregate Missing Fields Count CSV
    with open(
        field_summary_csv, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["missing_field_name", "missing_in_file_count"])
        for field_name, count in missing_fields_counter.most_common():
            writer.writerow([field_name, count])

    print(f"Scan complete for pattern '{pattern}':")
    print(f" - Critical Errors logged: {len(critical_errors)}")
    print(f" - Files with missing fields: {len(missing_fields_per_file)}")
    print(f" - Unique missing fields tracked: {len(missing_fields_counter)}")

    return (
        len(critical_errors),
        len(missing_fields_per_file),
        len(missing_fields_counter),
    )

In [ ]:
analyze_missing_fields_to_csv(
    root_directory="/home/hemduttdabral/Downloads/proj-api-volume-dump",
    schema="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_master_schema.json",
    critical_errors_csv="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_critical_errors.csv",
    missing_fields_csv="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_missing_fields.csv",
    field_summary_csv="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_missing_fields_summary.csv",
    filename_prefix="ai_")